# **Hospital Readmission Risk Modeling**
## Final Report

### Author:  Bhavana Sanghi

### Project Goal: Predict 30-day hospital readmission risk to support targeted intervention and care management.

### This notebook summarizes the final methodology, results, interpretation, and model selection decisions for the hospital readmission risk modeling project.

## Executive Summary

The objective of this project was to build a predictive model to identify patients at elevated risk of 30-day hospital readmission so that healthcare providers can target post-discharge interventions more effectively.

The modeling pipeline began with logistic regression baselines, followed by L1-regularized logistic regression (LASSO) for feature selection and interpretability, and then a tuned XGBoost model to capture nonlinear relationships. Model performance was evaluated using ROC-AUC, recall, precision, F1-score, lift/gains analysis, calibration, SHAP-based interpretability, and fairness metrics across demographic groups.

The final comparison showed that linear models achieved similar discrimination performance (test ROC-AUC around 0.662), while tuned XGBoost improved ranking performance modestly (test ROC-AUC 0.6675) and substantially increased recall. A reduced-feature XGBoost model, created by removing low-importance feature groups identified through SHAP analysis, retained essentially identical performance (test ROC-AUC 0.6673) with fewer features.

The final selected model is the **reduced XGBoost model**, because it preserves predictive performance while improving simplicity and deployment practicality. Lift analysis showed that the model effectively concentrates readmissions in the highest-risk deciles, and fairness analysis showed no severe bias, though moderate disparities in false negative rates should be monitored in deployment.

<font color="green"> Data cleaning and Preparation code - [Data cleaning and preparation](01_data_cleaning_prep.ipynb)</font>

<font color="green"> Modelling code - [Modelling](02_model_training_and_evaluation.ipynb)</font>


---

## Problem Framing

Hospital readmissions are costly for both patients and healthcare systems. An effective readmission risk model can support targeted intervention by identifying patients who may benefit from additional follow-up, discharge planning, or care coordination.

This project is framed as a **risk prioritization problem**, not only a binary classification problem. In practice, the goal is not simply to assign a yes/no label, but to rank patients by risk so that limited intervention resources can be directed toward the most vulnerable cases.

Because of this, the project emphasizes:

- **Recall**, to reduce missed high-risk patients
- **F1**, in some cases for simplicity of the project
- **ROC-AUC**, to measure ranking quality
- **Lift and gains**, to evaluate prioritization effectiveness
- **Interpretability**, to understand what drives predictions
- **Fairness**, to ensure performance is reasonably consistent across patient groups

---

## Dataset and Data Understanding

This project uses the **Diabetes 130-US Hospitals for Years 1999–2008** dataset from the UCI Machine Learning Repository. The dataset contains hospital encounter records for patients diagnosed with diabetes and was designed for studying early readmission, especially readmission within 30 days of discharge. According to UCI, the dataset represents **10 years (1999–2008) of clinical care at 130 US hospitals and integrated delivery networks**, with **101,766 encounters** and **47 original features**. The feature set includes demographic information, admission/discharge information, diagnosis codes, laboratory results, medication-related variables, and prior utilization counts such as inpatient, outpatient, and emergency visits.

The original prediction target in this dataset is readmission status, including whether a patient was readmitted within 30 days, after 30 days, or not readmitted. For this project, the problem was reframed as a **binary risk prediction task** focused on identifying patients likely to be readmitted within 30 days, because that aligns more directly with a clinical intervention use case.

The dataset is well suited to this problem because it contains several classes of information that are clinically relevant to readmission risk:

- **Demographics:** age, gender, race  
- **Encounter context:** admission type, admission source, discharge disposition, time in hospital  
- **Clinical burden:** diagnosis categories, number of diagnoses, number of medications, diabetic medication indicators  
- **Lab and treatment signals:** HbA1c / A1C-related features, diabetes medication changes  
- **Prior utilization:** counts of inpatient, outpatient, and emergency encounters before hospitalization  

### Dataset source

UCI Machine Learning Repository: **Diabetes 130-US Hospitals for Years 1999–2008** 

---

## Dataset Cleaning and feature Engineering methods - 

<font color="red">For detailed information on data cleaning and feature engineering please visit - [Data cleaning and preparation](01_data_cleaning_prep.ipynb)</font>

### 1. Target Definition

The original readmitted variable contained three categories: <30, >30, and NO. This was converted into a binary target:
1. 1 → readmitted within 30 days
2. 0 → not readmitted within 30 days

This transformation aligns the problem with a clinically actionable objective: identifying patients at risk of early readmission

### 2. Leakage-Aware Data Splitting 

To prevent information leakage, the dataset was split into training and test sets using patient-level grouping (patient_nbr). 
This ensures that all encounters for a given patient appear in only one split.
This step is critical because multiple encounters from the same patient can otherwise lead to overly optimistic performance estimate

### 3. Data Cleaning

### 3.1 Removing Ineligible Encounters

Encounters where the patient was discharged to hospice, died, or was transferred to a long-term inpatient setting were excluded, as readmission within 30 days is not a meaningful or fair outcome for these cases. Specifically, This removed 2,423 rows, leaving 99,343 encounters.

### 3.2 Handling Missing and Sparse Variables

The dataset used "?" as a placeholder for missing values across multiple columns. These were replaced with NaN to enable standard missing value handling.

Two columns were dropped entirely due to excessive missingness that made imputation impractical:

weight (~97% missing): the overwhelming absence of data made this column uninformative.

payer_code (~40% missing): dropped on the same grounds, and its inclusion would not directly improve clinical prediction.

### 3.3 Gender Cleaning

The gender column contained 3 records with the value "Unknown/Invalid". These were removed as they represented a negligible fraction of the data and could not be reliably encoded.

### 4. Target Distribution

- **Not readmitted within 30 days:** 88.6%
- **Readmitted within 30 days:** 11.4%

Class imbalance addressed through cost-sensitive learning
(scale_pos_weight = 7.74) — equivalent to fraud modeling
practice where positive class is substantially rarer than
negative class.

### 5. Feature Engineering 

### 5.0 Feature Grouping 

Before any transformations were applied, all 45 features were audited and assigned to one of nine groups based on their data type, cardinality, and the kind of preprocessing they would require. This grouping served as the blueprint for all subsequent engineering decisions.

1. Numeric (8 features): Continuous or discrete count variables representing clinical activity during the encounter or prior utilisation history — time_in_hospital, num_lab_procedures, num_procedures, num_medications, number_outpatient, number_emergency, number_inpatient, number_diagnoses.
2. Ordinal (1 feature): age, stored as ordered string buckets (e.g., [10-20), [20-30)), requiring conversion to a numeric ordinal representation before modelling.
3. Lab Results (2 features): max_glu_serum and A1Cresult — both categorical in nature, with high missingness interpreted as the test not having been ordered rather than a random data gap.
4. Diagnosis Codes (3 features): diag_1, diag_2, and diag_3, each containing hundreds of raw ICD-9 codes requiring mapping to broader clinical categories.
5. ID-Encoded Categoricals (3 features): admission_type_id, discharge_disposition_id, and admission_source_id — stored as integers whose numeric values carry no arithmetic meaning and needed to be decoded using the IDS_mapping.csv reference file.
6. Contextual Categorical (1 feature): medical_specialty — a high-cardinality categorical with ~49% missing values, requiring missingness imputation and category consolidation.
7. Medication Status (23 features): One column per diabetes medication (e.g., metformin, insulin, glipizide), each taking values of No, Steady, Up, or Down to indicate whether the drug was prescribed and whether the dosage changed during the encounter.
8. Binary (2 features): change (whether any diabetes medication was adjusted during the encounter) and diabetesMed (whether the patient was on any diabetes medication), both stored as Yes/No strings requiring simple binary encoding.
9. Nominal (2 features): race and gender — demographic attributes with no inherent ordering, retained primarily for subgroup analysis and fairness evaluation.

### 5.1 Age Encoding
The age column was stored as ordered string buckets (e.g., [10-20), [20-30), etc.). These were mapped to an ordinal numeric representation using midpoint encoding, making the feature suitable for both linear and tree-based models. No missing values were present in this column

### 5.2 Lab Result Features

1. max_glu_serum: Dropped due to ~95% missingness in the training set. The near-universal absence of this test result meant it carried insufficient signal.
2. A1Cresult: Missing values (~83%) were filled with "Not_Measured", reflecting the clinically meaningful interpretation that the test was simply not ordered rather than the result being unknown.

### 5.3 Diagnosis Code Mapping
The three diagnosis columns (diag_1, diag_2, diag_3) each contained over 700 unique ICD-9 codes. Following the methodology of Strack et al. (2014), these codes were mapped to nine broad clinical categories: Circulatory, Respiratory, Digestive, Genitourinary, Musculoskeletal, Neoplasms, Injury, Diabetes (all 250.xx codes), and Other (including V/E codes and unclassifiable entries).

An additional binary feature, diabetes_primary_diag, was engineered to flag encounters where diabetes was the primary diagnosis.

### 5.4 Prior Utilisation Features
The raw utilisation counts (number_outpatient, number_emergency, number_inpatient) were zero-inflated, so binary indicator flags were derived to capture whether any prior utilisation had occurred:

any_prior_outpatient, any_prior_emergency, any_prior_inpatient

A composite prior_utilization_score was also created as a weighted sum that reflects the relative clinical severity of each utilisation type:
prior_utilization_score = (number_inpatient × 3) + (number_emergency × 2) + (number_outpatient × 1)
This score showed a clear separation by outcome — the mean score for readmitted patients was roughly double that of non-readmitted patients, making it one of the stronger engineered signals.

### 5.5 Encounter Intensity Features
Two ratio features were derived to capture the intensity of care during the hospital stay, normalised by length of stay:

med_intensity = num_medications / time_in_hospital
lab_intensity = num_lab_procedures / time_in_hospital

These features distinguish patients who received a high volume of medications or lab tests in a short admission from those with a longer but less intensive stay.

### 5.6 Admission & Discharge Mapping
The three ID-encoded administrative variables were decoded from their raw integer codes using the IDS_mapping.csv reference file and collapsed into interpretable categories:

1. admission_type_id → admission_type: Emergency, Elective, Urgent, Unknown, Other 
2. admission_source_id → admission_source: Emergency Room, Physician Referral, Transfer, Other 
3. discharge_disposition_id → discharge_disposition: Home, Skilled Nursing, Home Health, Other, Transfer

A derived binary feature, institutional_discharge, was also created to flag patients discharged to a skilled nursing facility, rehabilitation, long-term care, or transferred to another hospital — outcomes likely associated with higher acuity and readmission risk. 

### 5.7 Medical Specialty Grouping

medical_specialty had missing values (~49% of training data). Missing and "?" values were filled with "Unknown". The remaining high-cardinality values (~70 specialties) were collapsed into 12 groups: Primary Care, Emergency, Cardiology, Surgery, Orthopedics, Nephrology, Pulmonology, Specialty Medicine, Mental Health, Radiology, Other, and Unknown.

### 5.8 Medication Features
Each of the 23 diabetes medication columns carried values of No, Steady, Up, or Down. Five aggregate features were derived from these:

1. n_active_diabetes_meds: count of medications with status Steady, Up, or Down
2. n_med_increases: count of medications with status Up
3. n_med_decreases: count of medications with status Down
4. insulin_flag: binary indicator for any insulin use (status not equal to No)
5. insulin_change: binary indicator for insulin dosage adjustment (Up or Down)

### 5.9 Binary & Nominal Features

1. change (medication change during encounter): encoded as binary (Ch = 1, No = 0). Patients with a medication change had a slightly higher readmission rate 
diabetesMed (on diabetes medication): encoded as binary (Yes = 1, No = 0). Patients on diabetes medication had a readmission rate of 11.9% vs. 10.0% for those not on medication 
race: missing values were filled with "Unknown". Readmission rates varied modestly across groups, from 8.3% (Unknown) to 11.6% (African American). The feature was retained for subgroup diagnostics.

gender: retained as a nominal binary feature after removing the 3 invalid records.

### 5.10 Summary of Feature Set
After cleaning and feature engineering, the final training set contained 79,538 encounters and 70 columns, of which 45 were original features and 25 were newly engineered. The final feature set used for modelling excluded the identifier columns (encounter_id, patient_nbr) and the original readmitted target, leaving 45+ engineered features entering the modelling pipeline.

---

## MODELLING 

<font color="red"> For more details on Modelling please refer - [Modelling](02_model_training_and_evaluation.ipynb)</font>

The modelling phase was approached as a structured progression from interpretable baselines through to a tuned, SHAP-validated gradient boosting model. A deliberate sequence was followed: establish a transparent logistic regression baseline, apply regularisation to identify the most informative features, then introduce XGBoost with systematic hyperparameter tuning and SHAP analysis-based feature reduction. 

the modeling strategy emphasized:
1. High recall (sensitivity) to minimize missed readmissions for non linear models and f1 metric (just for simplicity sake) in linear models
2. Ranking quality (ROC-AUC) to ensure effective prioritization
3. Operational usefulness (lift and gains) to support targeted intervention

The class imbalance in the dataset - approximately 11.4% positive (readmitted within 30 days) was addressed consistently across all models

**Level 1 — Training-time imbalance correction**

For logistic regression models, class_weight="balanced" was used to automatically rescale the loss function so the minority class receives proportionally higher weight during training. For XGBoost, the equivalent mechanism is scale_pos_weight, set to the ratio of negative to positive training examples, which achieves the same effect within the gradient boosting loss.

**Level 2 — Post-training threshold optimisation**

A threshold sweep was conducted on out-of-fold (OOF) predictions from cross-validation for every model. Two threshold strategies were evaluated: an F1-optimal threshold and a recall-oriented threshold, reflecting the clinical priority that missing a high-risk patient carries greater cost than an unnecessary flag. These two levels of imbalance handling were applied consistently across all model families.  

### Pre-Modelling: Multicollinearity and Feature Preparation

Before any model was trained, all categorical features were one-hot encoded expanded the feature space from 70 engineered columns to 83 model-ready columns

Post that, a correlation analysis and Variance Inflation Factor (VIF) check were conducted on the 16 numeric features. Three pairs of highly correlated variables were identified:

1. prior_utilization_score was highly correlate with number_inpatient
2. total_prior_visits was highly correlate with prior_utilization_score
3. total_prior_visits was highly correlate with number_inpatient

total_prior_visits was removed first as it was the most collinear. 

A follow-up VIF analysis confirmed residual multicollinearity among the raw count variables (number_emergency, number_inpatient, number_outpatient), which were also dropped in favour of their binary indicator counterparts (any_prior_inpatient, any_prior_emergency, any_prior_outpatient) and the composite prior_utilization_score. This reduced the numeric feature set from 16 to 12 and eliminated unstable coefficient estimation for logistic models.

For logistic regression, the 12 numeric features were standardised using StandardScaler fitted on the training set only and applied to the test set, yielding final matrices of shape (79,538 × 79) 

## 1. Logistic Regression Models
Unweighted Baseline (No Imbalance Correction)

A penalty-free logistic regression with no class weighting was trained first to establish the upper bound of linear separability and to demonstrate the effect of ignoring class imbalance. It achieved a CV AUC of 0.660 ± 0.002

To recover minority class performance, a **threshold sweep** was conducted on OOF cross-validation predictions across 200 evenly spaced thresholds from 0.01 to 0.99. The F1-optimal threshold was identified as 0.138 - far below 0.5, reflecting how far the predicted probabilities for positive cases were being suppressed without class weighting. At this threshold on the test set, the model achieved recall of 0.44 and precision of 0.20 (F1 = 0.27). 

## 2. Class-Weighted Logistic Regression
A logistic regression with class_weight="balanced" was then trained. 

This instructs scikit-learn to weight each class inversely proportional to its frequency during training, effectively upweighting every minority-class sample. The immediate effect was that the model's predicted probabilities spread more meaningfully across the probability range for both classes, as confirmed by the predicted probability distribution plot.
The balanced model achieved a CV AUC of 0.662 ± 0.003 — a marginal but consistent improvement over the unweighted baseline. Critically, the F1-optimal OOF threshold shifted to 0.542, far closer to the natural 0.5 boundary, indicating that class weighting had substantially corrected the probability scale distortion seen in the unweighted model. 

A KS statistic analysis was also conducted, identifying the threshold of maximum separation between the true positive rate and false positive rate curves as 0.471, with a KS statistic of 0.237. Performance was evaluated at the default 0.5, the F1-optimal (0.542), and the KS-optimal (0.471) thresholds, and compared across both logistic variants to guide model selection.


## 3. Lasso Logistic Regression (selected Logistic model)
Building on the class-weighted model, a regularisation path search was conducted over 10 logarithmically spaced values of C (from 0.001 to 100) for both L1 (Lasso) and L2 (Ridge) penalties, with class_weight="balanced" retained throughout. Each configuration was evaluated by 5-fold stratified CV AUC.
L1 regularisation at C = 0.046 achieved the best overall performance at CV AUC = 0.662 ± 0.003, marginally outperforming the best Ridge equivalent while also performing automatic feature selection. 

The fitted LASSO model zeroed out 17 of 79 features - including change, several race dummies, low-signal admission type and source categories, and minor diagnosis category indicators - leaving 62 active features.

Threshold tuning for the LASSO model followed the same OOF procedure. The F1-optimal threshold on OOF predictions was 0.530, at which the model achieved precision of 0.191 and recall of 0.488 (F1 = 0.275) on the OOF training data. 

The KS statistic on the test set was 0.238 at a threshold of 0.468. All three thresholds (default 0.50, KS-optimal 0.468, and F1-optimal 0.530) were evaluated on the test set and compiled into a unified comparison table.

The LASSO model was selected as the final logistic regression candidate: it matched the balanced model's discriminative performance, produced an interpretable sparse coefficient vector, and its implicit feature selection informed the subsequent XGBoost reduction step

## 4. Baseline XGBoost Model

An initial XGBoost classifier was trained with conservative defaults (n_estimators=1000, learning_rate=0.05, max_depth=3). Class imbalance was addressed via scale_pos_weight, computed as the ratio of negative to positive training examples. 

This is the XGBoost-native equivalent of class_weight="balanced" in scikit-learn, and directly scales the gradient contribution of positive-class examples during tree building. The baseline XGBoost achieved a CV AUC of 0.665 ± 0.003, already outperforming all logistic models.

## 4.1 Hyperparamter tuned XGBOOST Model

A RandomizedSearchCV was conducted over 30 random parameter combinations from a broad grid, with 5-fold stratified CV AUC as the scoring metric and scale_pos_weight held fixed throughout to preserve consistent imbalance handling across all configurations. The best configuration achieved a CV AUC of 0.670 ± 0.003, providing strong regularisation that prevents the model from overfitting to the small positive class signal.

## 4.2 Threshold tuned XGBOOST Model 

The same OOF threshold tuning methodology used for logistic regression was applied to the tuned XGBoost. OOF probabilities were generated using 5-fold stratified CV, and two threshold strategies were evaluated:

1. F1-optimal threshold: 0.55 — maximises F1 on OOF predictions.
2. Recall-oriented threshold: 0.44 — the highest recall achievable with precision ≥ 0.16, reflecting the clinical priority of minimising missed readmissions.

The recall-oriented threshold of 0.44 was adopted as the operating threshold. A precision floor of 16% was set to ensure the model remains actionable rather than flagging the majority of patients indiscriminately. All three thresholds were compared on the test set to validate the selection.

## 4.3 Feature Reduction via Permutation Importance & SHAP

To build a leaner, more deployable model without sacrificing performance, a two-stage importance analysis was conducted on the tuned XGBoost.

### Stage 1 — Permutation Importance.
Permutation importance was evaluated on the test set over 20 repeats, scored by AUC drop. This established which individual features had zero or negative predictive contribution and provided an initial signal about the model's most and least valuable inputs. 

The top drivers by permutation importance were prior_utilization_score, any_prior_inpatient, discharge_disposition_Home, and institutional_discharge — consistent with the clinical intuition that prior healthcare utilisation and post-discharge pathway are the strongest signals for readmission risk.

### Stage 2 — SHAP Grouped Importance Analysis. 

Rather than removing features one by one, SHAP values (computed using TreeExplainer on the test set) were aggregated into nine clinically meaningful feature groups: prior utilisation, discharge disposition, diagnosis, medical specialty, medication management, encounter intensity, demographics, admission context, and lab results. The mean absolute SHAP value was summed across all features within each group to quantify each group's total contribution to the model's predictions.

This analysis revealed that two groups — admission_context (covering all admission_type and admission_source dummy variables) and lab_results (the A1Cresult dummies) — collectively accounted for only approximately 2% of the model's total predictive power. Given this negligible contribution, all features belonging to these two groups were removed entirely, reducing the feature set from 79 to 69 features — a drop of 10 columns.

A SHAP beeswarm plot and a patient-level waterfall plot were also generated to further validate and communicate model behaviour. For a representative true positive (patient index 211, predicted probability 0.678, actual outcome: readmitted), the top risk drivers were a prior utilisation score of 6, a prior inpatient stay, institutional discharge, an 8-day length of stay, and 8 recorded diagnoses — a clinically coherent and interpretable risk profile.

### 4.4 Reduced XGBOOST Model
The reduced model was retrained using the identical tuned hyperparameters, with the recall-oriented OOF threshold procedure re-run on the reduced feature set. The optimal threshold shifted minimally from 0.44 to 0.45, and the test AUC moved from 0.6675 to 0.6673 — a reduction of just 0.0002 — confirming that the two dropped groups were genuinely redundant and that the simplified model was essentially equivalent in discriminative power.

### 4.5 Final Model Comparison 
The table below summarises the complete model progression. For each model, both the imbalance-handling strategy at training time and the threshold selection method are noted, reinforcing that discriminative performance (AUC) and operational performance (recall, precision, F1) are distinct dimensions that were addressed separately and deliberately.

<img src="../outputs/tables/final_models_comparison.png" width="800"/>

The XGBoost Reduced model was selected as the final candidate. It matches the full model's discriminative performance at a negligible AUC cost, uses a leaner validated feature set, and has been subjected to a complete evaluation pipeline: hyperparameter tuning, OOF threshold optimisation, SHAP-validated feature reduction, calibration assessment, and a structured fairness audit.

### 4.6 Model Calibration

A calibration curve was assessed for the final reduced XGBoost model using 10 equal-frequency bins. The model showed reasonable calibration at lower predicted probabilities but exhibited overconfidence at higher probability values — a known characteristic of gradient boosting models trained under class imbalance. No post-hoc recalibration (e.g., Platt scaling or isotonic regression) was applied at this stage, but this is identified as a consideration for production deployment where predicted probabilities are used directly for patient risk stratification or threshold adjustment.

<img src="../outputs/figures/calibration.png" width="800"/>

### 4.7 Fairness Analysis

A structured fairness audit was conducted on the final reduced XGBoost model at the recall-oriented operating threshold of 0.45, disaggregating AUC, recall, precision, F1, false negative rate (FNR), and false positive rate (FPR) across racial and gender subgroups on the held-out test set.

Across racial groups, recall ranged from 0.654 (Other) to 0.744 (Hispanic) — a spread of 9 percentage points. The corresponding FNR ranged from 25.6% (Hispanic) to 34.6% (Other), meaning the model misses a meaningfully higher proportion of actual readmissions for the "Other" racial group. The gender split showed more consistent performance across Female and Male subgroups. These disparities are documented transparently as a known limitation. Addressing them - for example through stratified threshold adjustment by subgroup, or fairness-aware retraining objectives - is identified as a priority for future model iterations.

<img src="../outputs/figures/fairness_race.png" width="800"/>



---

## MODEL UTILITY AND STRATIFIED RISK ANALYSIS

Beyond standard classification metrics, the true value of a readmission risk model lies in its operational utility. Specifically, how effectively it concentrates high-risk patients at the top of a ranked list so that limited clinical intervention resources can be allocated where they matter most. The lift and cumulative gains analysis on the held-out test set demonstrates that the final Reduced XGBoost model achieves this meaningfully

## 1. Decile-based lift analysis 

Patients in the test set were ranked by their predicted readmission probability and divided into ten equal-sized deciles, with Decile 1 representing the highest-risk patients. The table and cumulative gains chart below summarise model performance across deciles.

<img src="../outputs/tables/xgb_reduced_decile_table.png" width="800"/>


The results demonstrate strong and consistent risk stratification. Patients in the top decile (cum_positive_pct) — the 10% of encounters the model assigns highest risk — had an observed readmission rate of 26.8%, more than 2.4 times the overall population rate of 11.2%. This means that if clinical teams were to prioritise only the top-scoring 10% of patients for follow-up intervention, they would be working with a group where roughly 1 in 4 patients is genuinely at risk of readmission within 30 days.

The cumulative gains chart shows how this concentration of risk compounds across deciles

<img src="../outputs/figures/xgb_reduced_cumulative_chart.png" width="800"/>

By targeting the top 30% of patients by risk score, a clinical team would capture 50.5% of all readmissions — meaning half of all potentially avoidable readmissions can be addressed by intervening on only a third of the patient population. Extending to the top 50% captures 70.8% of all readmissions, at which point lift falls below 1.0, indicating the model has largely exhausted its ability to distinguish high-risk patients from the population average. This crossover at the 50th percentile provides a natural, data-driven boundary for defining the actionable intervention population in a deployment setting.

Compared to a random patient selection approach — represented by the diagonal baseline in the cumulative gains chart — the model's consistent separation above the baseline across all deciles confirms it delivers deployable ranking value. 

## 2. What the Model Learned: Clinical Coherence of Predictions

SHAP analysis was conducted on the full 79-feature tuned model as part of the feature selection process that produced the final reduced model. Interpretability conclusions are expected to hold for the final model given the near-identical predictive performance (AUC difference of 0.0002) and the negligible contribution of the removed feature groups to the model's predictive power.

The grouped SHAP importance chart shows that three clinical domains account for 64% of the model's total predictive power: prior healthcare utilisation (28%), discharge disposition (20%), and encounter intensity (16%). This is clinically coherent. A patient's history of prior inpatient admissions is one of the strongest known predictors of future hospitalisation - patients who have been admitted before are more likely to have chronic, complex conditions that drive repeated acute episodes. Discharge to a skilled nursing facility, rehabilitation centre, or long-term care institution signals that the patient was too unstable to return home, reflecting higher acuity and ongoing clinical need. Encounter intensity metrics - length of stay, number of medications, number of diagnoses, number of lab procedures - collectively capture how clinically complex the current admission was.

The SHAP beeswarm plot confirms directional consistency across all patients: high values of prior_utilization_score push predicted risk strongly upward; being discharged home consistently reduces predicted risk; and older patients are associated with elevated risk. None of these relationships are counterintuitive, which is a critical property for a model that clinical teams will be asked to act on and trust.
The patient-level waterfall plot illustrates this concretely. For the representative high-risk patient shown (predicted probability = 0.745, actual outcome: readmitted), the prediction was driven by a small set of highly interpretable factors: a prior utilisation score of 6 (+0.15), a confirmed prior inpatient stay (+0.15), institutional discharge (+0.12), not being discharged home (+0.09), an 8-day length of stay (+0.07), and 8 recorded diagnoses (+0.04). Starting from a population base rate of just 1.2%, these factors combined to push the predicted probability to 74.5% — and the patient was indeed readmitted. This is precisely the kind of transparent, auditable prediction chain that builds trust with clinical stakeholders and supports responsible deployment.

## 3. Summary
The final Reduced XGBoost model — trained with scale_pos_weight to handle class imbalance, tuned via randomised search across 30 hyperparameter configurations, validated through SHAP-guided feature reduction, and operated at a recall-oriented threshold of 0.45 — delivers a model that is simultaneously performant, interpretable, and operationally useful. It ranks patients effectively enough that targeting the top 30% by predicted risk captures more than half of all 30-day readmissions, while the features driving its predictions align closely with established clinical knowledge about readmission risk factors. The end-to-end pipeline — from patient-group-aware train/test splitting through to fairness auditing across racial and gender subgroups — was an attempt to delvier a production-conscious approach to developing a clinical risk model.
